# 03 – Frage 2: Mieten nach Kanton und Zimmerzahl

**Leitfrage:** Welche Kantone sind bei den Mieten Spitzenreiter, welche Nachzügler — und wo steht Zürich?

## Teil 1: Momentaufnahme 2024

Datensatz `je-d-09.03.03.05.xlsx` = durchschnittlicher Mietpreis **pro m²** nach Kanton und Zimmerzahl. **Wichtig:** Diese Datei enthält nur das Jahr 2024 — sie zeigt also das *Niveau*, nicht die Veränderung über die Zeit. Die zeitliche Entwicklung folgt in Teil 2.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RAW = "../data/raw"
PROCESSED = "../data/processed"

### Aufbereiten

Das Excel hat Titel-/Kopfzeilen und je Zimmerklasse zwei Spalten (Wert + Vertrauensintervall). Wir lesen ohne Header ein, nehmen nur die Wertspalten und filtern Fuss­noten weg (Zeilen ohne numerischen Total-Wert).

In [ ]:
raw = pd.read_excel(f"{RAW}/je-d-09.03.03.05.xlsx", header=None)

wert_spalten = [1, 3, 5, 7, 9, 11, 13]  # jede 2. Spalte ist das Vertrauensintervall
zimmer = ["Total", "1", "2", "3", "4", "5", "6+"]

zeilen = []
for i in range(5, raw.shape[0]):
    name = raw.iloc[i, 0]
    if not isinstance(name, str) or pd.isna(raw.iloc[i, 1]):
        continue  # Fussnoten / Leerzeilen
    rec = {"Kanton": name.strip()}
    for c, z in zip(wert_spalten, zimmer):
        rec[z] = pd.to_numeric(raw.iloc[i, c], errors="coerce")
    zeilen.append(rec)

df = pd.DataFrame(zeilen)
df.to_csv(f"{PROCESSED}/miete_kanton_zimmer_2024.csv", index=False)

ch = df[df["Kanton"] == "Schweiz"].iloc[0]
kantone = df[df["Kanton"] != "Schweiz"].copy()
print(f"{len(kantone)} Kantone | CH-Schnitt: {ch['Total']} Fr./m²")
df.head()

### Auswertung 1 – Ranking der Kantone (Total)

In [ ]:
d = kantone.sort_values("Total")
farben = ["#c8102e" if k == "Zürich" else "#5a7d99" for k in d["Kanton"]]

plt.figure(figsize=(8, 9))
plt.barh(d["Kanton"], d["Total"], color=farben)
plt.axvline(ch["Total"], color="gray", ls="--", lw=1)
plt.text(ch["Total"] + 0.1, 0.2, f"CH-Schnitt {ch['Total']}", color="gray", fontsize=9)
plt.title("Durchschnittsmiete pro m² nach Kanton (2024)\nZürich hervorgehoben")
plt.xlabel("Franken pro m² / Monat (Nettomiete)")
plt.tight_layout()
plt.savefig("../figures/miete_kanton_ranking_2024.png", dpi=130)
plt.show()

### Auswertung 2 – Heatmap Kanton × Zimmerzahl

In [ ]:
hm = kantone.set_index("Kanton")[["1", "2", "3", "4", "5", "6+"]].sort_values("3", ascending=False)

plt.figure(figsize=(8, 10))
sns.heatmap(hm, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "Fr./m²"})
plt.title("Miete pro m² nach Kanton und Zimmerzahl (2024)")
plt.xlabel("Zimmerzahl"); plt.ylabel("")
plt.tight_layout()
plt.savefig("../figures/miete_kanton_zimmer_heatmap_2024.png", dpi=130)
plt.show()

### Key Findings (Teil 1, 2024)

- **Teuerste Kantone:** Zug (21,5), **Zürich (21,3)**, Genf (21,1) Fr./m².
- **Günstigste Kantone:** Jura (12,3), Neuenburg (13,7), Appenzell A.Rh. (13,9).
- **Zürich liegt auf Rang 2 von 26** und rund **20 % über dem Schweizer Schnitt** (17,8).
- **Kleine Wohnungen sind pro m² am teuersten:** 1-Zimmer-Wohnungen kosten deutlich mehr pro m² als grosse — sichtbar als heller-zu-dunkel-Verlauf in der Heatmap.

## Teil 2: Entwicklung über die Zeit

_folgt — benötigt einen Mehrjahres-Datensatz aus der BFS-Datenbank (STAT-TAB)._